# Lahore Building Footprint Presence (Open Buildings)
## Export: 250 m aggregated point GeoJSON to Google Drive


### 0. Initialize Earth Engine


In [9]:
import ee
import geemap
import geopandas as gpd

EE_SCOPES = [
    "https://www.googleapis.com/auth/earthengine",
    "https://www.googleapis.com/auth/drive",
    "https://www.googleapis.com/auth/devstorage.full_control",
    "https://www.googleapis.com/auth/cloud-platform",
]
EE_PROJECT = "1035330553052"
FORCE_REAUTH = True  # Set to False after one successful Drive-authorized login.

if FORCE_REAUTH:
    ee.Authenticate(auth_mode="localhost", scopes=EE_SCOPES, force=True)

try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate(auth_mode="localhost", scopes=EE_SCOPES)
    ee.Initialize(project=EE_PROJECT)



Successfully saved authorization token.


### 1. Parameters


In [10]:
BOUNDARY_PATH = "../lahore.geojson"
YEAR = 2023
PRESENCE_THRESH = 0.5
EXPORT_SCALE_M = 250
OUT_PREFIX = f"building_footprint_lahore_openbuildings_{YEAR}"

if YEAR < 2016 or YEAR > 2023:
    raise ValueError("Open Buildings Temporal v1 is available only for annual snapshots from 2016 through 2023.")

print(f"Using Open Buildings annual snapshot: {YEAR}")
print(f"Output prefix: {OUT_PREFIX}")


Using Open Buildings annual snapshot: 2023
Output prefix: building_footprint_lahore_openbuildings_2023


### 2. Build Server-Side 250 m Grid and Aggregate Building Presence


In [11]:
gdf = gpd.read_file(BOUNDARY_PATH)[["geometry"]]
if gdf.crs is None:
    gdf = gdf.set_crs(4326)
if gdf.crs.to_epsg() != 4326:
    gdf = gdf.to_crs(4326)
gdf = gdf.dissolve().reset_index(drop=True)
gdf["geometry"] = gdf["geometry"].buffer(0)

region = geemap.gdf_to_ee(gdf).geometry()
grid_proj = ee.Projection("EPSG:32643").atScale(EXPORT_SCALE_M)
grid_fc = region.coveringGrid(grid_proj, EXPORT_SCALE_M)
grid_fc = grid_fc.map(
    lambda f: ee.Feature(
        f.geometry(),
        {
            "cell_id": f.id(),
            "area_ha": f.geometry().area(1).divide(10000),
        },
    )
)

collection = (
    ee.ImageCollection("GOOGLE/Research/open-buildings-temporal/v1")
    .filterBounds(region)
    .filterDate(f"{YEAR}-01-01", f"{YEAR + 1}-01-01")
)
template = ee.Image(collection.first())
presence_proj = template.select("building_presence").projection()
mosaic = collection.mosaic()
presence = mosaic.select("building_presence").setDefaultProjection(presence_proj).clip(region)
built = presence.gte(PRESENCE_THRESH).rename("built_share")

presence_stats = presence.reduceRegions(
    collection=grid_fc,
    reducer=ee.Reducer.mean(),
    scale=4,
    tileScale=4,
)
presence_stats = presence_stats.map(
    lambda f: ee.Feature(f.geometry(), f.toDictionary())
        .set("presence_mean", f.get("mean"))
        .select(["cell_id", "area_ha", "presence_mean"], None, False)
)

built_stats = built.reduceRegions(
    collection=grid_fc,
    reducer=ee.Reducer.mean(),
    scale=4,
    tileScale=4,
)
built_stats = built_stats.map(
    lambda f: ee.Feature(f.geometry(), f.toDictionary())
        .set("built_share", f.get("mean"))
        .select(["cell_id", "built_share"], None, False)
)

joined = ee.Join.inner().apply(
    presence_stats,
    built_stats,
    ee.Filter.equals(leftField="cell_id", rightField="cell_id"),
)

stats_fc = ee.FeatureCollection(
    joined.map(
        lambda pair: ee.Feature(
            ee.Feature(pair.get("primary")).geometry().centroid(1),
            ee.Feature(pair.get("primary")).toDictionary().combine(
                ee.Feature(pair.get("secondary")).toDictionary(),
                overwrite=True,
            ),
        )
    )
)


### 3. Export Aggregated Point GeoJSON to Google Drive


In [12]:
export_prefix = f"{OUT_PREFIX}_points_{EXPORT_SCALE_M}m"
task = ee.batch.Export.table.toDrive(
    collection=stats_fc,
    description=export_prefix,
    folder="EE_Exports",
    fileNamePrefix=export_prefix,
    fileFormat="GeoJSON",
)
task.start()

print(f"Started Drive export: EE_Exports/{export_prefix}.geojson")
print("Check task status in the Earth Engine Tasks tab or with task.status().")


Started Drive export: EE_Exports/building_footprint_lahore_openbuildings_2023_points_250m.geojson
Check task status in the Earth Engine Tasks tab or with task.status().
